# Sales & Order Management Analytics System

## Problem Statement

E-commerce businesses generate large amounts of customer, product,
order and payment data. Managing this information manually makes it
difficult to track sales, inventory and customer performance.

This project provides a centralized system to manage sales
transactions and generate useful business insights.

## Project Objective

The objective of this project is to develop an interactive Sales and
Order Management Analytics System that manages customers, products,
orders and payments while providing revenue, product and customer
analytics.

## Technology Selection

| Technology | Purpose |
|---|---|
| Python | Application logic and data processing |
| SQLite | Relational database |
| SQL | CRUD operations, joins and analytics |
| Pandas | Data analysis and tabular processing |
| Matplotlib | Data visualization |
| Gradio | Interactive web application |
| Google Colab | Development and execution environment |

## 1. Install Required Libraries

In [7]:
!pip -q install gradio pandas matplotlib

## 2. Import Libraries

In [8]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from datetime import date

DB_NAME = "sales_analytics.db"


## 3. Create Database and Tables

In [9]:
conn = sqlite3.connect(DB_NAME, check_same_thread=False)
cur = conn.cursor()

cur.executescript("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE,
    city TEXT,
    signup_date TEXT
);

CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL,
    category TEXT,
    price REAL NOT NULL,
    stock INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT,
    status TEXT,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE IF NOT EXISTS order_items (
    order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE IF NOT EXISTS payments (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    payment_date TEXT,
    amount REAL,
    payment_status TEXT,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);
""")

conn.commit()
print("Database and tables created successfully!")


Database and tables created successfully!


## 4. Insert Sample Data

In [10]:
# Insert sample data only if database is empty

customer_count = cur.execute("SELECT COUNT(*) FROM customers").fetchone()[0]

if customer_count == 0:
    customers = [
        ("Rahul", "rahul@gmail.com", "Chennai", "2026-01-10"),
        ("Priya", "priya@gmail.com", "Bangalore", "2026-01-15"),
        ("Arun", "arun@gmail.com", "Chennai", "2026-02-05"),
        ("Divya", "divya@gmail.com", "Coimbatore", "2026-02-12")
    ]

    products = [
        ("Laptop", "Electronics", 50000, 20),
        ("Headphones", "Electronics", 2000, 50),
        ("Keyboard", "Accessories", 1500, 40),
        ("Mouse", "Accessories", 800, 60),
        ("Monitor", "Electronics", 12000, 15)
    ]

    cur.executemany(
        "INSERT INTO customers(name,email,city,signup_date) VALUES (?,?,?,?)",
        customers
    )

    cur.executemany(
        "INSERT INTO products(product_name,category,price,stock) VALUES (?,?,?,?)",
        products
    )

    conn.commit()

    # Sample orders
    sample_orders = [
        (1, "2026-01-20", "COMPLETED", 52000),
        (2, "2026-01-25", "COMPLETED", 3500),
        (1, "2026-02-10", "COMPLETED", 2800),
        (3, "2026-02-15", "COMPLETED", 50000)
    ]

    cur.executemany(
        "INSERT INTO orders(customer_id,order_date,status,total_amount) VALUES (?,?,?,?)",
        sample_orders
    )

    # order_id, product_id, quantity, unit_price
    items = [
        (1, 1, 1, 50000),
        (1, 2, 1, 2000),
        (2, 2, 1, 2000),
        (2, 3, 1, 1500),
        (3, 2, 1, 2000),
        (3, 4, 1, 800),
        (4, 1, 1, 50000)
    ]

    cur.executemany(
        "INSERT INTO order_items(order_id,product_id,quantity,unit_price) VALUES (?,?,?,?)",
        items
    )

    payments = [
        (1, "2026-01-20", 52000, "PAID"),
        (2, "2026-01-25", 3500, "PAID"),
        (3, "2026-02-10", 2800, "PAID"),
        (4, "2026-02-15", 50000, "PAID")
    ]

    cur.executemany(
        "INSERT INTO payments(order_id,payment_date,amount,payment_status) VALUES (?,?,?,?)",
        payments
    )

    # Reduce stock for sample orders
    for _, product_id, qty, _ in items:
        cur.execute(
            "UPDATE products SET stock = stock - ? WHERE product_id = ?",
            (qty, product_id)
        )

    conn.commit()

print("Sample data ready!")


Sample data ready!


## 5. Application Functions

In [11]:
def get_customers():
    return pd.read_sql_query(
        "SELECT * FROM customers ORDER BY customer_id",
        conn
    )

def get_products():
    return pd.read_sql_query(
        "SELECT * FROM products ORDER BY product_id",
        conn
    )

def get_orders():
    query = """
    SELECT
        o.order_id,
        c.name AS customer,
        o.order_date,
        o.status,
        o.total_amount
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    ORDER BY o.order_id DESC
    """
    return pd.read_sql_query(query, conn)

def get_payments():
    query = """
    SELECT
        p.payment_id,
        p.order_id,
        c.name AS customer,
        p.payment_date,
        p.amount,
        p.payment_status
    FROM payments p
    JOIN orders o ON o.order_id = p.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    ORDER BY p.payment_id DESC
    """
    return pd.read_sql_query(query, conn)


def add_customer(name, email, city):
    if not name.strip():
        return "Please enter customer name.", get_customers()

    try:
        cur.execute(
            """
            INSERT INTO customers(name,email,city,signup_date)
            VALUES (?,?,?,?)
            """,
            (name.strip(), email.strip(), city.strip(), str(date.today()))
        )
        conn.commit()
        return "Customer added successfully!", get_customers()
    except sqlite3.IntegrityError:
        return "Email already exists.", get_customers()


def add_product(product_name, category, price, stock):
    if not product_name.strip():
        return "Please enter product name.", get_products()

    try:
        price = float(price)
        stock = int(stock)

        if price < 0 or stock < 0:
            return "Price and stock must be positive.", get_products()

        cur.execute(
            """
            INSERT INTO products(product_name,category,price,stock)
            VALUES (?,?,?,?)
            """,
            (product_name.strip(), category.strip(), price, stock)
        )
        conn.commit()
        return "Product added successfully!", get_products()
    except:
        return "Enter valid price and stock values.", get_products()


def customer_choices():
    df = get_customers()
    return [f"{r.customer_id} - {r.name}" for _, r in df.iterrows()]


def product_choices():
    df = get_products()
    return [f"{r.product_id} - {r.product_name} (Stock: {r.stock})" for _, r in df.iterrows()]


def refresh_choices():
    return (
        gr.update(choices=customer_choices()),
        gr.update(choices=product_choices())
    )


def place_order(customer_value, product_value, quantity):
    if not customer_value or not product_value:
        return "Select customer and product.", get_orders(), get_products()

    try:
        customer_id = int(customer_value.split(" - ")[0])
        product_id = int(product_value.split(" - ")[0])
        quantity = int(quantity)

        if quantity <= 0:
            return "Quantity must be greater than 0.", get_orders(), get_products()

        product = cur.execute(
            "SELECT product_name, price, stock FROM products WHERE product_id=?",
            (product_id,)
        ).fetchone()

        if product is None:
            return "Product not found.", get_orders(), get_products()

        product_name, price, stock = product

        if quantity > stock:
            return f"Only {stock} units available.", get_orders(), get_products()

        total_amount = price * quantity
        today = str(date.today())

        cur.execute(
            """
            INSERT INTO orders(customer_id,order_date,status,total_amount)
            VALUES (?,?,?,?)
            """,
            (customer_id, today, "COMPLETED", total_amount)
        )

        order_id = cur.lastrowid

        cur.execute(
            """
            INSERT INTO order_items(order_id,product_id,quantity,unit_price)
            VALUES (?,?,?,?)
            """,
            (order_id, product_id, quantity, price)
        )

        cur.execute(
            "UPDATE products SET stock = stock - ? WHERE product_id = ?",
            (quantity, product_id)
        )

        cur.execute(
            """
            INSERT INTO payments(order_id,payment_date,amount,payment_status)
            VALUES (?,?,?,?)
            """,
            (order_id, today, total_amount, "PAID")
        )

        conn.commit()

        message = (
            f"Order #{order_id} placed successfully! "
            f"{product_name} x {quantity} = ₹{total_amount:,.2f}"
        )

        return message, get_orders(), get_products()

    except Exception as e:
        return f"Error: {e}", get_orders(), get_products()


def dashboard_metrics():
    total_revenue = cur.execute(
        """
        SELECT COALESCE(SUM(total_amount),0)
        FROM orders
        WHERE status='COMPLETED'
        """
    ).fetchone()[0]

    total_orders = cur.execute(
        "SELECT COUNT(*) FROM orders"
    ).fetchone()[0]

    total_customers = cur.execute(
        "SELECT COUNT(*) FROM customers"
    ).fetchone()[0]

    total_products = cur.execute(
        "SELECT COUNT(*) FROM products"
    ).fetchone()[0]

    avg_order = cur.execute(
        """
        SELECT COALESCE(AVG(total_amount),0)
        FROM orders
        WHERE status='COMPLETED'
        """
    ).fetchone()[0]

    return (
        f"₹{total_revenue:,.2f}",
        str(total_orders),
        str(total_customers),
        str(total_products),
        f"₹{avg_order:,.2f}"
    )


def monthly_revenue():
    query = """
    SELECT
        substr(order_date,1,7) AS month,
        SUM(total_amount) AS revenue
    FROM orders
    WHERE status='COMPLETED'
    GROUP BY substr(order_date,1,7)
    ORDER BY month
    """
    return pd.read_sql_query(query, conn)


def top_products():
    query = """
    SELECT
        p.product_name,
        SUM(oi.quantity) AS units_sold,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.product_name
    ORDER BY revenue DESC
    LIMIT 10
    """
    return pd.read_sql_query(query, conn)


def customer_ranking():
    query = """
    SELECT
        c.name AS customer,
        COUNT(o.order_id) AS total_orders,
        SUM(o.total_amount) AS revenue
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.status='COMPLETED'
    GROUP BY c.customer_id, c.name
    ORDER BY revenue DESC
    """
    return pd.read_sql_query(query, conn)


def sales_chart():
    df = monthly_revenue()

    fig, ax = plt.subplots(figsize=(8,4))

    if len(df) > 0:
        ax.bar(df["month"], df["revenue"])
        ax.set_xlabel("Month")
        ax.set_ylabel("Revenue")
        ax.set_title("Monthly Revenue")
        ax.tick_params(axis="x", rotation=45)
    else:
        ax.text(0.5, 0.5, "No sales data", ha="center", va="center")

    plt.tight_layout()
    return fig


def refresh_dashboard():
    revenue, orders, customers, products, avg = dashboard_metrics()
    return (
        revenue,
        orders,
        customers,
        products,
        avg,
        monthly_revenue(),
        top_products(),
        customer_ranking(),
        sales_chart()
    )


## 6. Build Interactive Gradio Application

In [12]:
with gr.Blocks(title="Sales Analytics System") as app:

    gr.Markdown(
        """
        # 🛒 Sales & Order Management Analytics System
        ### SQL + Python Data Analytics Project
        """
    )

    with gr.Tab("📊 Dashboard"):

        refresh_btn = gr.Button("Refresh Dashboard")

        with gr.Row():
            revenue_box = gr.Textbox(label="Total Revenue", interactive=False)
            orders_box = gr.Textbox(label="Total Orders", interactive=False)
            customers_box = gr.Textbox(label="Customers", interactive=False)

        with gr.Row():
            products_box = gr.Textbox(label="Products", interactive=False)
            avg_box = gr.Textbox(label="Average Order Value", interactive=False)

        revenue_plot = gr.Plot(label="Monthly Revenue")

        gr.Markdown("### Monthly Revenue")
        monthly_table = gr.Dataframe(interactive=False)

        gr.Markdown("### Top Products")
        top_table = gr.Dataframe(interactive=False)

        gr.Markdown("### Customer Ranking")
        ranking_table = gr.Dataframe(interactive=False)

        refresh_btn.click(
            refresh_dashboard,
            outputs=[
                revenue_box,
                orders_box,
                customers_box,
                products_box,
                avg_box,
                monthly_table,
                top_table,
                ranking_table,
                revenue_plot
            ]
        )

    with gr.Tab("👤 Customers"):

        customer_msg = gr.Textbox(label="Status", interactive=False)

        with gr.Row():
            customer_name = gr.Textbox(label="Name")
            customer_email = gr.Textbox(label="Email")
            customer_city = gr.Textbox(label="City")

        add_customer_btn = gr.Button("Add Customer")

        customers_table = gr.Dataframe(
            value=get_customers(),
            label="Customers",
            interactive=False
        )

        add_customer_btn.click(
            add_customer,
            inputs=[
                customer_name,
                customer_email,
                customer_city
            ],
            outputs=[
                customer_msg,
                customers_table
            ]
        )

    with gr.Tab("📦 Products"):

        product_msg = gr.Textbox(label="Status", interactive=False)

        with gr.Row():
            product_name = gr.Textbox(label="Product Name")
            product_category = gr.Textbox(label="Category")
            product_price = gr.Number(label="Price")
            product_stock = gr.Number(label="Stock", precision=0)

        add_product_btn = gr.Button("Add Product")

        products_table = gr.Dataframe(
            value=get_products(),
            label="Products",
            interactive=False
        )

        add_product_btn.click(
            add_product,
            inputs=[
                product_name,
                product_category,
                product_price,
                product_stock
            ],
            outputs=[
                product_msg,
                products_table
            ]
        )

    with gr.Tab("🛍️ Place Order"):

        gr.Markdown(
            "Select a customer and product, enter quantity, then place the order."
        )

        load_choices_btn = gr.Button("Load / Refresh Customer & Product Lists")

        customer_dropdown = gr.Dropdown(
            choices=customer_choices(),
            label="Customer"
        )

        product_dropdown = gr.Dropdown(
            choices=product_choices(),
            label="Product"
        )

        order_quantity = gr.Number(
            label="Quantity",
            value=1,
            precision=0
        )

        place_order_btn = gr.Button("Place Order")

        order_msg = gr.Textbox(label="Order Status", interactive=False)

        orders_table = gr.Dataframe(
            value=get_orders(),
            label="Orders",
            interactive=False
        )

        order_products_table = gr.Dataframe(
            value=get_products(),
            label="Current Product Stock",
            interactive=False
        )

        load_choices_btn.click(
            refresh_choices,
            outputs=[
                customer_dropdown,
                product_dropdown
            ]
        )

        place_order_btn.click(
            place_order,
            inputs=[
                customer_dropdown,
                product_dropdown,
                order_quantity
            ],
            outputs=[
                order_msg,
                orders_table,
                order_products_table
            ]
        )

    with gr.Tab("💳 Payments"):

        payment_refresh = gr.Button("Refresh Payments")

        payment_table = gr.Dataframe(
            value=get_payments(),
            label="Payments",
            interactive=False
        )

        payment_refresh.click(
            get_payments,
            outputs=payment_table
        )

    with gr.Tab("📈 SQL Analytics"):

        gr.Markdown("### Monthly Revenue")
        analytic_monthly = gr.Dataframe(
            value=monthly_revenue(),
            interactive=False
        )

        gr.Markdown("### Top Selling Products")
        analytic_products = gr.Dataframe(
            value=top_products(),
            interactive=False
        )

        gr.Markdown("### High Value Customers")
        analytic_customers = gr.Dataframe(
            value=customer_ranking(),
            interactive=False
        )

        analytics_refresh = gr.Button("Refresh Analytics")

        analytics_refresh.click(
            lambda: (
                monthly_revenue(),
                top_products(),
                customer_ranking()
            ),
            outputs=[
                analytic_monthly,
                analytic_products,
                analytic_customers
            ]
        )

    app.load(
        refresh_dashboard,
        outputs=[
            revenue_box,
            orders_box,
            customers_box,
            products_box,
            avg_box,
            monthly_table,
            top_table,
            ranking_table,
            revenue_plot
        ]
    )

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4adf6291b0d0b3f497.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. What to Show During Demo

1. Open the **Dashboard** and explain total revenue, orders and average order value.
2. Go to **Customers** and add a new customer.
3. Go to **Products** and add a new product.
4. Open **Place Order**, select customer/product and place an order.
5. Return to **Dashboard** and click **Refresh Dashboard**.
6. Show that revenue and stock changed automatically.
7. Open **SQL Analytics** and explain top products and customer ranking.

This demonstrates database creation, CRUD, relationships, joins, aggregations, analytics and reporting.


## Individual Contribution

I designed the database structure and relationships, created the
SQL queries, implemented the Python application, developed the
customer, product, order and payment workflows, implemented the
analytics functions, tested the application and prepared the
project presentation.